In [1]:
spark.stop()

NameError: name 'spark' is not defined

In [2]:
%run utils.py

In [3]:
from utils import *
spark = instantiate_spark_sedona("10g")


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/02/02 15:37:48 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/02/02 15:37:49 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


Spark Session and SedonaContext have been successfully initiated.


## Assigning subdistricts to thai quadkeys

In [5]:
quadkeys = spark.read.parquet("s3a://propheus-data-staging/Thailand/Gee_Buildings/quadkey.parquet")
quadkeys = quadkeys.distinct()
quadkeys.count()

1606681

In [6]:
quadkeys.show()

+----------------+--------------------+
|         quadkey|    quadkey_geometry|
+----------------+--------------------+
|1322033110233322|POLYGON ((100.700...|
|1322033101132002|POLYGON ((100.458...|
|1322033112002122|POLYGON ((100.568...|
|1322033101110230|POLYGON ((100.469...|
|1322031331223220|POLYGON ((100.942...|
|1322031332330311|POLYGON ((100.848...|
|1322031330200113|POLYGON ((100.585...|
|1322031332002031|POLYGON ((100.563...|
|1322031323131300|POLYGON ((100.524...|
|1322031321223111|POLYGON ((100.277...|
|1322031330000303|POLYGON ((100.574...|
|1322031321013301|POLYGON ((100.354...|
|1322031312112003|POLYGON ((100.816...|
|1322031311230030|POLYGON ((100.997...|
|1322031312000103|POLYGON ((100.574...|
|1322031312000120|POLYGON ((100.568...|
|1322031313001112|POLYGON ((100.975...|
|1322120200012130|POLYGON ((101.370...|
|1322031303020211|POLYGON ((100.211...|
|1322031300031311|POLYGON ((100.014...|
+----------------+--------------------+
only showing top 20 rows



In [7]:
from pyspark.sql.functions import *
quadkeys = quadkeys.withColumn('quadkey_geometry', expr("ST_GeomFromWKT(quadkey_geometry)"))
quadkeys.printSchema()

root
 |-- quadkey: string (nullable = true)
 |-- quadkey_geometry: geometry (nullable = true)



In [8]:
quadkeys_with_centroid = quadkeys.withColumn("quadkey_centroid",expr("ST_Centroid(quadkey_geometry)"))
quadkeys_with_centroid.show()

+----------------+--------------------+--------------------+
|         quadkey|    quadkey_geometry|    quadkey_centroid|
+----------------+--------------------+--------------------+
|1322033110233322|POLYGON ((100.700...|POINT (100.703430...|
|1322033101132002|POLYGON ((100.458...|POINT (100.461730...|
|1322033112002122|POLYGON ((100.568...|POINT (100.571594...|
|1322033101110230|POLYGON ((100.469...|POINT (100.472717...|
|1322031331223220|POLYGON ((100.942...|POINT (100.945129...|
|1322031332330311|POLYGON ((100.848...|POINT (100.851745...|
|1322031330200113|POLYGON ((100.585...|POINT (100.588073...|
|1322031332002031|POLYGON ((100.563...|POINT (100.566101...|
|1322031323131300|POLYGON ((100.524...|POINT (100.527648...|
|1322031321223111|POLYGON ((100.277...|POINT (100.280456...|
|1322031330000303|POLYGON ((100.574...|POINT (100.577087...|
|1322031321013301|POLYGON ((100.354...|POINT (100.357360...|
|1322031312112003|POLYGON ((100.816...|POINT (100.818786...|
|1322031311230030|POLYGO

In [9]:
quadkeys_with_centroid = quadkeys_with_centroid.distinct()
quadkeys_with_centroid.count()

1606681

In [11]:
adm3_geom = spark.read.parquet('admin3_geom')
adm3_geom.printSchema()

root
 |-- adm3_name: string (nullable = true)
 |-- adm2_name: string (nullable = true)
 |-- adm1_name: string (nullable = true)
 |-- geometry: geometry (nullable = true)



In [16]:
quad_keys_w_geom = quadkeys_with_centroid.alias("a").join(
    adm3_geom.alias("b"),
    expr("ST_Contains(b.geometry, a.quadkey_centroid)"),
    "inner"
)                                  

In [15]:
quad_keys_w_geom = spark.read.parquet("quadkeys_w_subdistrict")  
quad_keys_w_geom.show()

[Stage 53:>                                                         (0 + 1) / 1]

+----------------+--------------------+--------------------+---------+---------+------------+--------------------+
|         quadkey|    quadkey_geometry|    quadkey_centroid|adm3_name|adm2_name|   adm1_name|            geometry|
+----------------+--------------------+--------------------+---------+---------+------------+--------------------+
|1322031031303121|POLYGON ((99.7393...|POINT (99.7421264...|  Lat Yao|  Lat Yao|Nakhon Sawan|POLYGON ((99.7644...|
|1322031031303130|POLYGON ((99.7448...|POINT (99.7476196...|  Lat Yao|  Lat Yao|Nakhon Sawan|POLYGON ((99.7644...|
|1322031031303131|POLYGON ((99.7503...|POINT (99.7531127...|  Lat Yao|  Lat Yao|Nakhon Sawan|POLYGON ((99.7644...|
|1322031031303102|POLYGON ((99.7338...|POINT (99.7366333...|  Lat Yao|  Lat Yao|Nakhon Sawan|POLYGON ((99.7644...|
|1322031031303103|POLYGON ((99.7393...|POINT (99.7421264...|  Lat Yao|  Lat Yao|Nakhon Sawan|POLYGON ((99.7644...|
|1322031031303112|POLYGON ((99.7448...|POINT (99.7476196...|  Lat Yao|  Lat Yao|

In [18]:
quad_keys_w_geom.select(col('quadkey')).distinct().count() ,quad_keys_w_geom.count()


26/01/31 10:16:13 WARN JoinQuery: UseIndex is true, but no index exists. Will build index on the fly.
26/01/31 10:16:36 WARN JoinQuery: UseIndex is true, but no index exists. Will build index on the fly.
                                                                                

(1492664, 1492664)

In [25]:
quadkey_list = quad_keys_w_geom.select(col('quadkey')).distinct().rdd.map(lambda r: r[0]).collect()
len(quadkey_list)
# quadkey_list
# quadkeys_with_centroid.filter(col('quadkeys').isnotin(quad_keys_w_geom.select(col('quadkey'))))

26/01/31 10:24:25 WARN JoinQuery: UseIndex is true, but no index exists. Will build index on the fly.
                                                                                ]

1492664

In [27]:
quadkeys_not_in_geom = quadkeys_with_centroid.alias("a").join(
    quad_keys_w_geom.select("quadkey").distinct().alias("b"),
    col("a.quadkey") == col("b.quadkey"),
    "left_anti"
)
quadkeys_not_in_geom.persist()
quadkeys_not_in_geom.count()

26/01/31 10:26:38 WARN JoinQuery: UseIndex is true, but no index exists. Will build index on the fly.
                                                                                

114017

In [39]:
quadkeys_not_in_geom.select(col('quadkey')).distinct().count()

114017

In [35]:
quadkeys_not_in_geom.printSchema()

root
 |-- quadkey: string (nullable = true)
 |-- quadkey_geometry: geometry (nullable = true)
 |-- quadkey_centroid: geometry (nullable = true)



In [36]:
quad_keys_w_geom_intersects = quadkeys_not_in_geom.alias("a").join(
    adm3_geom.alias("b"),
    expr("ST_INTERSECTS(b.geometry, a.quadkey_geometry)"),
    "inner"
)
quad_keys_w_geom_intersects.count()

26/01/31 10:34:52 WARN JoinQuery: UseIndex is true, but no index exists. Will build index on the fly.
                                                                                

12485

In [ ]:
quad_keys_w_geom_intersects

In [60]:
quad_keys_w_geom_intersects_nondup_quadkeys.show()

26/01/31 11:04:01 WARN JoinQuery: UseIndex is true, but no index exists. Will build index on the fly.
[Stage 1028:===================================================>(328 + 1) / 329]

+----------------+
|         quadkey|
+----------------+
|1322213023132212|
|1322213023132121|
|1322213023132021|
|1322213023132210|
|1322213023132221|
|1322213023132022|
|1322213023132031|
|1322213023132003|
|1322213023132101|
|1322213023132011|
|1322213023132220|
|1322213023123313|
|1322213023132032|
|1322213023123311|
|1322213023132120|
|1322213023132020|
|1322213023132012|
|1322213212020300|
|1322213212022120|
|1322213212022012|
+----------------+
only showing top 20 rows



In [61]:
quad_keys_w_geom_intersects_nondup_quadkeys = quad_keys_w_geom_intersects.groupBy(col('quadkey')).agg(count('*').alias('count')).filter(col('count') == 1).select(col('quadkey'))

quad_keys_w_geom_intersects_nondups = quad_keys_w_geom_intersects.join(
    quad_keys_w_geom_intersects_nondup_quadkeys,
    quad_keys_w_geom_intersects.quadkey == quad_keys_w_geom_intersects_nondup_quadkeys.quadkey,
    "inner"
).select(quad_keys_w_geom_intersects['*'])
quad_keys_w_geom_intersects_nondups.count()

26/01/31 11:05:49 WARN JoinQuery: UseIndex is true, but no index exists. Will build index on the fly.
26/01/31 11:05:50 WARN JoinQuery: UseIndex is true, but no index exists. Will build index on the fly.
                                                                                ]

11639

In [69]:
quad_keys_w_geom_intersects_dup_quadkeys = quad_keys_w_geom_intersects.groupBy(col('quadkey')).agg(count('*').alias('count')).filter(col('count') > 1).select(col('quadkey'))
quad_keys_w_geom_intersects_dups = quad_keys_w_geom_intersects.join(
    quad_keys_w_geom_intersects_dup_quadkeys,
    quad_keys_w_geom_intersects.quadkey == quad_keys_w_geom_intersects_dup_quadkeys.quadkey,
    "inner"
).select(quad_keys_w_geom_intersects['*'])
quad_keys_w_geom_intersects_dup_quadkeys.count()

26/01/31 11:16:24 WARN JoinQuery: UseIndex is true, but no index exists. Will build index on the fly.
                                                                                

420

In [71]:
quad_keys_w_geom_intersects_dups.show()

26/01/31 11:16:51 WARN JoinQuery: UseIndex is true, but no index exists. Will build index on the fly.
26/01/31 11:16:53 WARN JoinQuery: UseIndex is true, but no index exists. Will build index on the fly.
[Stage 1575:===================================================>(327 + 5) / 332]

+----------------+--------------------+--------------------+-------------+-------------+---------+--------------------+
|         quadkey|    quadkey_geometry|    quadkey_centroid|    adm3_name|    adm2_name|adm1_name|            geometry|
+----------------+--------------------+--------------------+-------------+-------------+---------+--------------------+
|1322213021123120|POLYGON ((99.0307...|POINT (99.0335083...| Ko Lanta Yai|     Ko Lanta|    Krabi|MULTIPOLYGON (((9...|
|1322213021123120|POLYGON ((99.0307...|POINT (99.0335083...|     Sala Dan|     Ko Lanta|    Krabi|MULTIPOLYGON (((9...|
|1322213032112310|POLYGON ((99.4372...|POINT (99.4400024...|     Na Kluea|      Kantang|    Trang|MULTIPOLYGON (((9...|
|1322213032112310|POLYGON ((99.4372...|POINT (99.4400024...|    Ko Libong|      Kantang|    Trang|MULTIPOLYGON (((9...|
|1322213033002313|POLYGON ((99.5306...|POINT (99.5333862...|  Kantang Tai|      Kantang|    Trang|POLYGON ((99.5076...|
|1322213033002313|POLYGON ((99.5306...|P

In [76]:
from pyspark.sql.window import *

quad_keys_w_geom_intersects_dups = quad_keys_w_geom_intersects_dups.withColumn(
    "intersection_geom",
    expr("ST_Intersection(quadkey_geometry, geometry)")
).withColumn(
    "intersection_area",
    expr("ST_Area(intersection_geom)")
)
w = Window.partitionBy("quadkey").orderBy(col("intersection_area").desc())

quad_keys_w_geom_intersects_dups = quad_keys_w_geom_intersects_dups.withColumn(
    "rank", row_number().over(w)
)

quad_keys_w_geom_intersects_dups_deduped = quad_keys_w_geom_intersects_dups.filter(col('rank') == 1)\
    .select(col('quadkey'), col('quadkey_geometry'), col('quadkey_centroid'), col('adm3_name'), col('adm2_name'), col('adm1_name'), col('geometry'))
quad_keys_w_geom_intersects_dups_deduped.count()


26/01/31 11:48:09 WARN JoinQuery: UseIndex is true, but no index exists. Will build index on the fly.
26/01/31 11:48:11 WARN JoinQuery: UseIndex is true, but no index exists. Will build index on the fly.
                                                                                00]

420

In [ ]:
final_quad_keys_w_geom_df

In [80]:

final_quad_keys_w_geom_df = quad_keys_w_geom.unionByName(quad_keys_w_geom_intersects_nondups).unionByName(quad_keys_w_geom_intersects_dups_deduped)
# final_quad_keys_w_geom_df.write.mode("overwrite").parquet("abfss://propheus-data-science@propheusdatabay.dfs.core.windows.net/Thailand/thailand_quadkeys/thailand_quadkeys_w_subdistrict") 

In [6]:
from pyspark.sql.functions import *
# final_quad_keys_w_geom_df = spark.read.parquet("abfss://propheus-data-science@propheusdatabay.dfs.core.windows.net/Thailand/thailand_quadkeys/thailand_quadkeys_w_subdistrict") 
final_quad_keys_w_geom_names = final_quad_keys_w_geom_df.select(col('quadkey'), col('adm3_name'), col('adm2_name'), col('adm1_name'))
final_quad_keys_w_geom_names.show()

+----------------+-----------------+------------+---------+
|         quadkey|        adm3_name|   adm2_name|adm1_name|
+----------------+-----------------+------------+---------+
|1322122012022123|Thung Mahacharoen|Wang Nam Yen|  Sa Kaeo|
|1322122012022132|Thung Mahacharoen|Wang Nam Yen|  Sa Kaeo|
|1322122012022133|Thung Mahacharoen|Wang Nam Yen|  Sa Kaeo|
|1322122012023022|Thung Mahacharoen|Wang Nam Yen|  Sa Kaeo|
|1322122012023023|Thung Mahacharoen|Wang Nam Yen|  Sa Kaeo|
|1322122012023020|Thung Mahacharoen|Wang Nam Yen|  Sa Kaeo|
|1322122012023021|Thung Mahacharoen|Wang Nam Yen|  Sa Kaeo|
|1322122012023030|Thung Mahacharoen|Wang Nam Yen|  Sa Kaeo|
|1322122012023120|Thung Mahacharoen|Wang Nam Yen|  Sa Kaeo|
|1322122012023121|Thung Mahacharoen|Wang Nam Yen|  Sa Kaeo|
|1322122012023130|Thung Mahacharoen|Wang Nam Yen|  Sa Kaeo|
|1322122012023102|Thung Mahacharoen|Wang Nam Yen|  Sa Kaeo|
|1322122012023103|Thung Mahacharoen|Wang Nam Yen|  Sa Kaeo|
|1322122012023100|Thung Mahacharoen|Wang

In [8]:
# final_quad_keys_w_geom_names.toPandas().to_csv('quadkeys_w_admin_names.csv',index=False)

In [2]:
import pandas as pd
final_quad_keys_w_geom_names = pd.read_csv('quadkeys_w_admin_names.csv')
final_quad_keys_w_geom_names.count()

quadkey      1504723
adm3_name    1504723
adm2_name    1504723
adm1_name    1504723
dtype: int64

In [37]:
quadkeys_not_in_geom_intersects = quadkeys_not_in_geom.alias("a").join(
    quad_keys_w_geom_intersects.select("quadkey").distinct().alias("b"),
    col("a.quadkey") == col("b.quadkey"),
    "left_anti"
)
quadkeys_not_in_geom_intersects.persist()
quadkeys_not_in_geom_intersects.count()

26/01/31 10:36:35 WARN JoinQuery: UseIndex is true, but no index exists. Will build index on the fly.
                                                                                

101958

In [ ]:
# quadkeys_not_in_geom_intersects.toPandas().to_csv('quadkeys_not_in_geom_intersects.csv', index=False)

## Quadkey level HH estimation